# Round 1. OLTP-style queries

In [0]:
%sql
SELECT order_id, order_status, order_purchase_timestamp
FROM orders_2025
LIMIT 5;

5 rows came back

In [0]:
%sql
SELECT *
FROM orders_2025
WHERE order_id = '720fd9e9-cdeb-4dec-a187-f71586eb085a';

1 row came back

Someone trying to get the details of their order would ask this question in a real shop

# Round 2. OLAP-style queries

In [0]:
%sql
SELECT date_format(order_purchase_timestamp, 'yyyy-MM') AS order_month,
order_status,
COUNT(*) AS total_orders
FROM orders_2025
GROUP BY order_month, order_status
ORDER BY order_month, order_status;

The query read 283,573 rows and 24 rows came back

Round 1 dealt with non-aggregated data while round 2 dealt with aggregated data.

Business Stakeholder who has to make some strategic decisions or has to look into the overall current business scenario. 

# Round 3. History in Delta Lake

In [0]:
%sql
CREATE TABLE orders_2025_copy AS
SELECT * FROM orders_2025;

In [0]:
%sql
SELECT order_id, order_status
FROM orders_2025_copy
WHERE order_status = 'delivered'
LIMIT 1;

In [0]:
%sql
UPDATE orders_2025_copy
SET order_status = 'canceled'
WHERE order_id = 'e6441daf-f23c-4a2e-9c72-90403032e1b6';

In [0]:
%sql
SELECT order_id, order_status
FROM orders_2025_copy
WHERE order_id = 'e6441daf-f23c-4a2e-9c72-90403032e1b6';

In [0]:
%sql
DESCRIBE HISTORY orders_2025_copy;

In [0]:
%sql
SELECT order_id, order_status
FROM orders_2025_copy VERSION AS OF 0
WHERE order_id = 'e6441daf-f23c-4a2e-9c72-90403032e1b6';

order_status was updated to 'canceled' when the update command was run.

Version 0 still holds the old value

An ordinary OLTP system would have lost the history of changes made in the databse.

# Leaderboard Challenge

In [0]:
%sql
SELECT date_format(order_purchase_timestamp, 'EEEE') AS day_of_week,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
       ROUND(SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct
FROM orders_2025
WHERE order_purchase_timestamp >= '2025-01-01'
  AND order_purchase_timestamp <  '2025-12-01'
GROUP BY day_of_week
ORDER BY cancellation_rate_pct DESC;

In [0]:
%sql
SELECT date_format(order_purchase_timestamp, 'yyyy-MM') AS order_month,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
       ROUND(SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct
FROM orders_2025
WHERE order_purchase_timestamp >= '2025-01-01'
  AND order_purchase_timestamp <  '2026-01-01'
GROUP BY order_month
ORDER BY order_month;

In [0]:
%sql
SELECT date_format(order_purchase_timestamp, 'EEEE') AS day_of_week,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
       ROUND(SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct
FROM orders_2025
WHERE order_purchase_timestamp >= '2025-12-01'
  AND order_purchase_timestamp <  '2026-01-01'
GROUP BY day_of_week
ORDER BY cancellation_rate_pct DESC;

The cancellation rate of December is above 31% which is way above the avg of other months about 4% to 6%. 

Looking into Quartiles,

Q1 = 4.77%

Q3 = 5.03%

IQR = Q3 - Q1 = 0.25

Fence for outlier 

Lower fence : Q1 - 1.5IQR = 4.395

Upper fence : Q3 + 1.5IQR = 5.395

Cancellation rate of december is 31.16% which is way above the upper fence of 5.395%.